# 00d — TheSession Irish Folk: Data Preparation

**Thesis context:** TheSession.org is a community database of Irish traditional music
published under CC BY-NC 4.0. The data export contains ABC notation for over 267,000
tune *settings* (individual notations of tunes). A setting is one person's written version
of a tune; many settings exist for the same underlying tune.

**Pipeline:**
1. Deduplicate by `tune_id` (keep one setting per unique tune)
2. Join with `tune_popularity.csv` to rank by community popularity
3. Stratify selection across tune types (reel, jig, hornpipe, etc.) for repertoire diversity
4. Convert ABC notation → MIDI via `music21`

**Output:**
- `data/processed/irish_folk/midi/` — 200 MIDI files
- `data/metadata/irish_folk_tunes.csv` — per-tune metadata

**Key metadata used:**
- `type`: tune category (reel, jig, hornpipe, polka, slide, waltz, mazurka, strathspey, ...)
- `mode`: key and mode (e.g. "Gmajor", "Ador", "Dmixolydian")
- `meter`: time signature (4/4, 6/8, 9/8, 3/4, ...)
- `tune_popularity.csv`: tunebooks count — proxy for community canonical status


In [1]:
import sys
from pathlib import Path

# Locate project root regardless of where Jupyter was launched from.
# Searches upward for PROGRESS.md — the root marker.
_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
import pandas as pd
import re
import time
import warnings

from music21 import converter, midi as m21_midi

RAW_CSV  = PROJECT_ROOT / "datasets" / "irish_folk" / "TheSession-data" / "csv"
OUT_MIDI = PROJECT_ROOT / "data" / "processed" / "irish_folk" / "midi"
META_DIR = PROJECT_ROOT / "data" / "metadata"

OUT_MIDI.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

/Users/mohammadashraf/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load tunes.csv and inspect

In [3]:
tunes = pd.read_csv(RAW_CSV / "tunes.csv")
print(f"Total rows (settings): {len(tunes):,}")
print(f"Columns: {tunes.columns.tolist()}")
print(f"Unique tune_ids: {tunes['tune_id'].nunique():,}")
tunes.head(3)

Total rows (settings): 54,681
Columns: ['tune_id', 'setting_id', 'name', 'type', 'meter', 'mode', 'abc', 'date', 'username', 'composer']
Unique tune_ids: 23,103


,tune_id,setting_id,name,type,meter,mode,abc,date,username,composer
0,15326,28560,'S Ann An Ìle,strathspey,4/4,Gmajor,|:G>A B>G c>A B>G|E<E A>G F<D D2|G>A B>G c>A B...,2016-03-31 15:34:45,danninagh,NaN
1,15326,28582,'S Ann An Ìle,strathspey,4/4,Gmajor,"uD2|:{F}v[G,2G2]uB>ud c>A B>G|{D}E2 uA>uG F<D ...",2016-04-03 09:15:08,DonaldK,NaN
2,14625,26955,'S Daor An Tabac,reel,4/4,Bminor,|:eAAB eABB|eAAB gedB|eAAB eABB|G2AB gedB:|\r\...,2015-07-31 02:47:47,Charles Mackenzie,NaN


In [4]:
print("Tune type distribution (all settings):")
print(tunes["type"].value_counts().head(15).to_string())
print("\nMode distribution (top 10):")
print(tunes["mode"].value_counts().head(10).to_string())
print("\nMeter distribution:")
print(tunes["meter"].value_counts().to_string())

Tune type distribution (all settings):
type
reel          19937
jig           13835
polka          4200
waltz          3942
hornpipe       3743
slip jig       1960
march          1802
barndance      1735
strathspey     1359
slide          1163
mazurka         586
three-two       419

Mode distribution (top 10):
mode
Gmajor         14342
Dmajor         14188
Amajor          3909
Adorian         3293
Eminor          3088
Edorian         2585
Bminor          2126
Amixolydian     1806
Aminor          1733
Dmixolydian     1590

Meter distribution:
meter
4/4     28576
6/8     13835
3/4      4528
2/4      4200
9/8      1960
12/8     1163
3/2       419


## 2. Load popularity and deduplicate

In [5]:
popularity = pd.read_csv(RAW_CSV / "tune_popularity.csv")
print(f"Popularity entries: {len(popularity):,}")
popularity.head(3)

Popularity entries: 11,684


,name,tune_id,tunebooks
0,Drowsy Maggie,27,7811
1,"Kesh, The",55,7556
2,Cooley's,1,6657


In [6]:
# Keep one setting per tune_id — the lowest setting_id is typically the
# oldest/original notation. This gives us one ABC string per unique tune.
unique_tunes = (
    tunes
    .sort_values("setting_id")
    .drop_duplicates(subset="tune_id", keep="first")
    .copy()
)
print(f"Unique tunes after deduplication: {len(unique_tunes):,}")

# Join with popularity
unique_tunes = unique_tunes.merge(
    popularity[["tune_id", "tunebooks"]],
    on="tune_id",
    how="left"
)
unique_tunes["tunebooks"] = unique_tunes["tunebooks"].fillna(0).astype(int)
print(f"Tunes with popularity data: {unique_tunes['tunebooks'].gt(0).sum():,}")
print(f"\nTop 5 tunes by popularity:")
print(unique_tunes.nlargest(5, "tunebooks")[["name", "type", "mode", "tunebooks"]].to_string())

Unique tunes after deduplication: 23,103
Tunes with popularity data: 11,683

Top 5 tunes by popularity:
              name      type     mode  tunebooks
25   Drowsy Maggie      reel  Edorian       7811
53       Kesh, The       jig   Gmajor       7556
0         Cooley's      reel  Edorian       6657
9   Butterfly, The  slip jig   Eminor       6189
69      Morrison's       jig  Edorian       5956


## 3. Stratified selection — 200 tunes across type categories

We sample proportionally from each tune type so the selection reflects the natural
distribution of the Irish tradition (which is dominated by reels and jigs).


In [7]:
N = 200

# Focus on the main tune types with enough data
main_types = unique_tunes["type"].value_counts()
print("Type distribution in unique tunes:")
print(main_types.to_string())

Type distribution in unique tunes:
type
reel          8011
jig           5983
waltz         2021
polka         1709
hornpipe      1574
slip jig       793
march          778
barndance      709
strathspey     624
slide          470
mazurka        253
three-two      178


In [8]:
# Sort by popularity within each type, then sample proportionally
sampled_parts = []
type_counts = unique_tunes["type"].value_counts()
total_unique = len(unique_tunes)

for tune_type, count in type_counts.items():
    n_select = max(1, round(N * count / total_unique))
    subset = (
        unique_tunes[unique_tunes["type"] == tune_type]
        .sort_values("tunebooks", ascending=False)
        .head(n_select)
    )
    sampled_parts.append(subset)

sampled = pd.concat(sampled_parts)
if len(sampled) > N:
    sampled = sampled.nlargest(N, "tunebooks")
elif len(sampled) < N:
    remaining = unique_tunes[~unique_tunes["tune_id"].isin(sampled["tune_id"])]
    topup = remaining.nlargest(N - len(sampled), "tunebooks")
    sampled = pd.concat([sampled, topup])

sampled = sampled.reset_index(drop=True)
print(f"Selected {len(sampled)} tunes")
print("\nType distribution in selection:")
print(sampled["type"].value_counts().to_string())

Selected 200 tunes

Type distribution in selection:
type
reel          69
jig           52
waltz         17
polka         15
hornpipe      14
slip jig       7
march          7
barndance      6
strathspey     5
slide          4
mazurka        2
three-two      2


## 4. ABC → MIDI conversion

In [9]:
def mode_to_abc_key(mode_str):
    """Convert TheSession mode strings to ABC K: field notation.

    Examples: 'Gmajor' → 'G', 'Aminor' → 'Am', 'Dmixolydian' → 'Dmix'
    Reference: ABC notation standard v2.1
    """
    mode_map = [
        ("major", ""),
        ("minor", "m"),
        ("mixolydian", "mix"),
        ("dorian", "dor"),
        ("lydian", "lyd"),
        ("phrygian", "phr"),
        ("locrian", "loc"),
    ]
    s = str(mode_str)
    for suffix, abc_suffix in mode_map:
        if s.lower().endswith(suffix):
            key_letter = s[:len(s) - len(suffix)]
            return key_letter + abc_suffix
    return s  # fallback: use as-is


def build_abc_string(row):
    """Construct a complete ABC notation string from a TheSession tunes.csv row."""
    key = mode_to_abc_key(row["mode"])
    meter = str(row["meter"])
    note_len = "1/8"  # standard default
    title = str(row["name"]).replace('"', "'")
    composer = str(row.get("composer", "")) if pd.notna(row.get("composer", "")) else ""

    header  = f"X:{row['tune_id']}\n"
    header += f"T:{title}\n"
    if composer:
        header += f"C:{composer}\n"
    header += f"M:{meter}\n"
    header += f"L:{note_len}\n"
    header += f"K:{key}\n"
    return header + str(row["abc"])


# Test on first tune
test_row = sampled.iloc[0]
test_abc = build_abc_string(test_row)
print("Sample ABC string:")
print(test_abc[:300])

Sample ABC string:
X:27
T:Drowsy Maggie
M:4/4
L:1/8
K:Edor
|:E2BE dEBE|E2BE AFDF|E2BE dEBE|BABc dAFD:|
d2fd c2ec|defg afge|d2fd c2ec|BABc dAFA|
d2fd c2ec|defg afge|afge fdec|BABc dAFD|


In [10]:
midi_paths = []
failed     = []
warnings.filterwarnings("ignore")  # suppress music21 verbose output

for i, row in sampled.iterrows():
    type_safe = str(row["type"]).replace(" ", "_")
    out_name  = f"irish_{i:03d}_{row['tune_id']}_{type_safe}.mid"
    out_path  = OUT_MIDI / out_name

    if out_path.exists():
        print(f"[{i+1:3d}/{len(sampled)}] SKIP  {out_name}")
        midi_paths.append(str(out_path))
        continue

    try:
        abc_str = build_abc_string(row)
        score   = converter.parse(abc_str, format="abc")
        score.write("midi", fp=str(out_path))
        midi_paths.append(str(out_path))
        if (i + 1) % 25 == 0:
            print(f"[{i+1:3d}/{len(sampled)}] Converted {i+1} tunes...")
    except Exception as e:
        failed.append({"index": i, "tune_id": row["tune_id"], "error": str(e)})
        midi_paths.append(None)

print(f"\n=== Conversion complete ===")
print(f"Converted : {sum(p is not None for p in midi_paths)}/{len(sampled)}")
print(f"Failed    : {len(failed)}")
if failed:
    for f in failed[:10]:
        print(f"  tune_id={f['tune_id']}: {f['error']}")

[  1/200] SKIP  irish_000_27_reel.mid
[  2/200] SKIP  irish_001_1_reel.mid
[  3/200] SKIP  irish_002_182_reel.mid
[  4/200] SKIP  irish_003_64_reel.mid
[  5/200] SKIP  irish_004_8_reel.mid
[  6/200] SKIP  irish_005_116_reel.mid
[  7/200] SKIP  irish_006_248_reel.mid
[  8/200] SKIP  irish_007_73_reel.mid
[  9/200] SKIP  irish_008_42_reel.mid
[ 10/200] SKIP  irish_009_98_reel.mid
[ 11/200] SKIP  irish_010_68_reel.mid
[ 12/200] SKIP  irish_011_103_reel.mid
[ 13/200] SKIP  irish_012_74_reel.mid
[ 14/200] SKIP  irish_013_197_reel.mid
[ 15/200] SKIP  irish_014_20_reel.mid
[ 16/200] SKIP  irish_015_72_reel.mid
[ 17/200] SKIP  irish_016_208_reel.mid
[ 18/200] SKIP  irish_017_113_reel.mid
[ 19/200] SKIP  irish_018_2_reel.mid
[ 20/200] SKIP  irish_019_69_reel.mid
[ 21/200] SKIP  irish_020_118_reel.mid
[ 22/200] SKIP  irish_021_75_reel.mid
[ 23/200] SKIP  irish_022_221_reel.mid
[ 24/200] SKIP  irish_023_517_reel.mid
[ 25/200] SKIP  irish_024_589_reel.mid
[ 26/200] SKIP  irish_025_141_reel.mid
[ 2

[ 35/200] SKIP  irish_034_518_reel.mid
[ 36/200] SKIP  irish_035_222_reel.mid
[ 37/200] SKIP  irish_036_86_reel.mid
[ 38/200] SKIP  irish_037_114_reel.mid
[ 39/200] SKIP  irish_038_399_reel.mid
[ 40/200] SKIP  irish_039_87_reel.mid
[ 41/200] SKIP  irish_040_430_reel.mid
[ 42/200] SKIP  irish_041_646_reel.mid
[ 43/200] SKIP  irish_042_138_reel.mid
[ 44/200] SKIP  irish_043_115_reel.mid
[ 45/200] SKIP  irish_044_18_reel.mid
[ 46/200] SKIP  irish_045_432_reel.mid
[ 47/200] SKIP  irish_046_791_reel.mid
[ 48/200] SKIP  irish_047_570_reel.mid
[ 49/200] SKIP  irish_048_857_reel.mid
[ 50/200] SKIP  irish_049_726_reel.mid
[ 51/200] SKIP  irish_050_188_reel.mid
[ 52/200] SKIP  irish_051_105_reel.mid
[ 53/200] SKIP  irish_052_5270_reel.mid
[ 54/200] SKIP  irish_053_99_reel.mid
[ 55/200] SKIP  irish_054_219_reel.mid
[ 56/200] SKIP  irish_055_602_reel.mid
[ 57/200] SKIP  irish_056_605_reel.mid
[ 58/200] SKIP  irish_057_1977_reel.mid
[ 59/200] SKIP  irish_058_2549_reel.mid
[ 60/200] SKIP  irish_059_

[126/200] SKIP  irish_125_957_waltz.mid
[127/200] SKIP  irish_126_601_waltz.mid
[128/200] SKIP  irish_127_562_waltz.mid
[129/200] SKIP  irish_128_790_waltz.mid
[130/200] SKIP  irish_129_1292_waltz.mid
[131/200] SKIP  irish_130_1016_waltz.mid
[132/200] SKIP  irish_131_1055_waltz.mid
[133/200] SKIP  irish_132_4906_waltz.mid
[134/200] SKIP  irish_133_3690_waltz.mid
[135/200] SKIP  irish_134_187_waltz.mid


[137/200] SKIP  irish_136_986_waltz.mid
[138/200] SKIP  irish_137_1678_waltz.mid
[139/200] SKIP  irish_138_441_polka.mid
[140/200] SKIP  irish_139_1075_polka.mid
[141/200] SKIP  irish_140_39_polka.mid
[142/200] SKIP  irish_141_238_polka.mid
[143/200] SKIP  irish_142_291_polka.mid
[144/200] SKIP  irish_143_357_polka.mid
[145/200] SKIP  irish_144_85_polka.mid
[146/200] SKIP  irish_145_481_polka.mid
[147/200] SKIP  irish_146_239_polka.mid
[148/200] SKIP  irish_147_583_polka.mid
[149/200] SKIP  irish_148_2434_polka.mid
[150/200] SKIP  irish_149_418_polka.mid
[151/200] SKIP  irish_150_531_polka.mid
[152/200] SKIP  irish_151_1529_polka.mid
[153/200] SKIP  irish_152_466_polka.mid
[154/200] SKIP  irish_153_83_hornpipe.mid
[155/200] SKIP  irish_154_475_hornpipe.mid
[156/200] SKIP  irish_155_49_hornpipe.mid
[157/200] SKIP  irish_156_651_hornpipe.mid
[158/200] SKIP  irish_157_30_hornpipe.mid
[159/200] SKIP  irish_158_566_hornpipe.mid
[160/200] SKIP  irish_159_13_hornpipe.mid
[161/200] SKIP  irish

[180/200] SKIP  irish_179_706_march.mid
[181/200] SKIP  irish_180_1863_march.mid
[182/200] SKIP  irish_181_1307_barndance.mid
[183/200] SKIP  irish_182_1615_barndance.mid
[184/200] SKIP  irish_183_2067_barndance.mid
[185/200] SKIP  irish_184_444_barndance.mid
[186/200] SKIP  irish_185_4577_barndance.mid
[187/200] SKIP  irish_186_1376_barndance.mid
[188/200] SKIP  irish_187_3016_strathspey.mid
[189/200] SKIP  irish_188_1746_strathspey.mid
[190/200] SKIP  irish_189_647_strathspey.mid
[191/200] SKIP  irish_190_170_strathspey.mid
[193/200] SKIP  irish_192_250_slide.mid
[194/200] SKIP  irish_193_70_slide.mid
[195/200] SKIP  irish_194_53_slide.mid
[196/200] SKIP  irish_195_1398_slide.mid
[197/200] SKIP  irish_196_5476_mazurka.mid
[198/200] SKIP  irish_197_2320_mazurka.mid
[199/200] SKIP  irish_198_2081_three-two.mid
[200/200] SKIP  irish_199_5151_three-two.mid

=== Conversion complete ===
Converted : 196/200
Failed    : 4
  tune_id=454: the object (<music21.meter.TimeSignature 3/4>, id()=478

## 5. Save metadata and verify

In [11]:
sampled = sampled.copy()
sampled["midi_path"] = midi_paths
sampled.to_csv(META_DIR / "irish_folk_tunes.csv", index=False)

midi_files = list(OUT_MIDI.glob("*.mid"))
print(f"MIDI files in output dir : {len(midi_files)}")
print(f"Conversion success rate  : {sampled['midi_path'].notna().sum()}/{len(sampled)}")
print(f"\nMode distribution in selection:")
print(sampled["mode"].value_counts().head(10).to_string())
print("\n✓ Irish folk preparation complete.")

MIDI files in output dir : 196
Conversion success rate  : 196/200

Mode distribution in selection:
mode
Dmajor         61
Gmajor         48
Adorian        27
Edorian        17
Eminor         11
Dmixolydian     8
Bminor          7
Amajor          7
Amixolydian     6
Aminor          4

✓ Irish folk preparation complete.
